# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL, 브랜치 이름, GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    branch_name = input("Git 브랜치 이름 (예: main / 기본 브랜치면 Enter): ").strip()
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        clone_cmd = ["git", "clone"]
        if branch_name:
            clone_cmd.extend(["-b", branch_name])
        clone_cmd.extend([clone_url, str(repo_dir)])
        subprocess.run(clone_cmd, check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")
        if branch_name:
            subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
            subprocess.run(["git", "checkout", branch_name], cwd=repo_dir, check=True)
            subprocess.run(["git", "pull", "--ff-only", "origin", branch_name], cwd=repo_dir, check=True)

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
current_branch = subprocess.run(["git", "branch", "--show-current"], cwd=repo_dir, text=True, capture_output=True).stdout.strip()
print(f"Repo: {repo_dir}")
print(f"Branch: {current_branch or '(detached HEAD)'}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

## 10. BPE vocab 생성/로드

10번 셀은 `nsmc_bpe_vocab_3000.json`을 먼저 찾아서 그대로 로드합니다. 찾는 순서는 Google Drive, repo의 `data/`, 현재 폴더, Colab `/content`입니다.

파일이 없으면 `corpus[:1_500_000]`과 `vocab_size=3000`으로 새로 학습해 repo `data/`와 가능한 경우 Google Drive에 저장합니다. 따라서 11번 셀은 항상 이 셀에서 준비된 `tokenizer`를 사용합니다.


In [ ]:
# 10. BPE vocab 생성/로드
import shutil
import sys
import time
from pathlib import Path

from bpe import BPETokenizer


# 과제 Basic 기준: corpus[:1_500_000], vocab_size=3000
VOCAB_SIZE = 3000
BPE_TRAIN_CHARS = 1_500_000
VOCAB_FILENAME = f"nsmc_bpe_vocab_{VOCAB_SIZE}.json"
VOCAB_PATH = Path(repo_dir) / "data" / VOCAB_FILENAME

# Colab에서는 로컬에서 만든 vocab을 올려서 쓰는 흐름이 기본입니다.
# 로컬에서는 파일이 없으면 학습하고, Colab에서는 파일이 없으면 업로드 안내를 냅니다.
IN_COLAB = "google.colab" in sys.modules
TRAIN_BPE_IF_MISSING = True
FORCE_RETRAIN_BPE = False
DRIVE_VOCAB_PATH = Path("/content/drive/MyDrive/W14-gpt-lab/data") / VOCAB_FILENAME if IN_COLAB else None
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print(f"Drive vocab path: {DRIVE_VOCAB_PATH}")


def find_vocab_file() -> Path | None:
    """Drive, repo data/, Colab /content에서 vocab 후보를 찾습니다."""
    candidates = [
        DRIVE_VOCAB_PATH,
        VOCAB_PATH,
        Path.cwd() / VOCAB_FILENAME,
        Path("/content") / VOCAB_FILENAME,
    ]
    for candidate in candidates:
        if candidate is not None:
            candidate = Path(candidate)
            if candidate.exists():
                return candidate
    return None


if "corpus" not in globals() or not corpus:
    LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
    if not LM_TRAIN_PATH.exists():
        raise FileNotFoundError("먼저 2번 NSMC 데이터 준비 셀을 실행하세요.")
    corpus = LM_TRAIN_PATH.read_text(encoding="utf-8")

start_time = time.time()
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)
existing_vocab = find_vocab_file()

if existing_vocab is not None and not FORCE_RETRAIN_BPE:
    tokenizer.load(existing_vocab)
    if len(tokenizer.id_to_token) != VOCAB_SIZE:
        message = f"vocab 크기 불일치: {existing_vocab} has {len(tokenizer.id_to_token)}, expected {VOCAB_SIZE}"
        if TRAIN_BPE_IF_MISSING:
            print(message)
            print("로컬이므로 Basic vocab을 새로 학습합니다.")
            existing_vocab = None
        else:
            raise ValueError(message + "입니다. Basic용 vocab 파일을 업로드하세요.")
    else:
        # Colab에 /content/nsmc_bpe_vocab_3000.json처럼 올린 경우 repo data/로 복사해
        # 11번 셀도 같은 경로에서 안정적으로 찾게 합니다.
        VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
        if existing_vocab.resolve() != VOCAB_PATH.resolve():
            shutil.copy2(existing_vocab, VOCAB_PATH)
            print(f"업로드 vocab 복사: {existing_vocab} -> {VOCAB_PATH}")
        print(f"기존 vocab 로드: {existing_vocab}")

if existing_vocab is None and (TRAIN_BPE_IF_MISSING or FORCE_RETRAIN_BPE):
    train_text = corpus[:BPE_TRAIN_CHARS]
    print(f"BPE 학습 시작: chars={len(train_text):,}, vocab_size={VOCAB_SIZE}")
    tokenizer.train(train_text)
    tokenizer.save(VOCAB_PATH)
    print(f"새 vocab 저장: {VOCAB_PATH}")
    if DRIVE_VOCAB_PATH is not None:
        DRIVE_VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
        tokenizer.save(DRIVE_VOCAB_PATH)
        print(f"Drive vocab 저장: {DRIVE_VOCAB_PATH}")
elif existing_vocab is None:
    raise FileNotFoundError(
        "Colab에서는 로컬에서 만든 vocab 파일을 먼저 업로드하세요. "
        f"권장 파일명: {VOCAB_FILENAME}"
    )

print("vocab_size 설정:", tokenizer.vocab_size)
print("실제 token 수:", len(tokenizer.id_to_token))
print("merge 수:", len(tokenizer.merges))
print(f"소요 시간: {time.time() - start_time:.1f}초")

sample = "이 영화는 좋았다"
sample_ids = tokenizer.encode(sample, add_bos_eos=True)
print("sample ids:", sample_ids[:20])
print("decode:", tokenizer.decode(sample_ids))

## 11. Basic 사전 학습 실행

10번에서 로드하거나 생성한 BPE vocab을 사용해 참조 브랜치와 같은 Basic 설정으로 mini GPT를 사전 학습합니다. 기본값은 `vocab_size=3000`, `context_length=128`, `emb_dim=128`, `n_layers=2`, `batch_size=8`, `num_epochs=3`입니다.

Colab/Drive가 있으면 token id cache, checkpoint, loss curve, summary JSON을 `MyDrive/W14-gpt-lab/` 아래에 저장하고, 로컬에서는 `cache/`, `checkpoints/`, `runs/`를 사용합니다.


In [ ]:
# 11. Basic 사전 학습 실행
import json
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from dataset import create_dataloader
from model import GPTModel
from train import train_model, calc_loss_loader

cell_start_time = time.perf_counter()

# 10번 셀에서 vocabulary를 load/create한 tokenizer를 그대로 사용합니다.
# tokenizer가 없다면 먼저 BPE vocabulary load 셀을 실행하세요.
assert "tokenizer" in globals(), "먼저 BPE vocabulary load 셀을 실행해 tokenizer를 준비하세요."
assert len(tokenizer.id_to_token) == 3000, f"expected vocab size 3000, got {len(tokenizer.id_to_token)}"
assert "corpus" in globals() and len(corpus) > 0, "먼저 NSMC LM train corpus를 로드하세요."

# 전체 corpus로 최종 사전 학습을 시작합니다.
# 빠른 탐색이 필요하면 TRAIN_CHAR_LIMIT를 300_000 또는 500_000으로 낮춰서 먼저 비교하세요.
TRAIN_CHAR_LIMIT = None
VAL_CHAR_LIMIT = None

train_text = corpus if TRAIN_CHAR_LIMIT is None else corpus[:TRAIN_CHAR_LIMIT]
if "val_corpus" in globals() and val_corpus:
    val_text = val_corpus if VAL_CHAR_LIMIT is None else val_corpus[:VAL_CHAR_LIMIT]
else:
    split_idx = int(len(train_text) * 0.9)
    val_text = train_text[split_idx:]
    train_text = train_text[:split_idx]

context_length = 128
batch_size = 8
learning_rate = 3e-4
num_epochs = 3
eval_freq = 100
eval_iter = 20
ckpt_freq = 1
encode_each_line_with_bos_eos = True

config = {
    "vocab_size": len(tokenizer.id_to_token),
    "context_length": context_length,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

def encode_lm_text(text: str, add_bos_eos_per_line: bool) -> list[int]:
    if not add_bos_eos_per_line:
        return tokenizer.encode(text)

    token_ids = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            token_ids.extend(tokenizer.encode(line, add_bos_eos=True))
    return token_ids


cache_root = Path("/content/drive/MyDrive/W14-gpt-lab/cache")
if not cache_root.exists():
    cache_root = Path("cache")
cache_root.mkdir(parents=True, exist_ok=True)

bos_tag = "bos" if encode_each_line_with_bos_eos else "nobos"
train_limit_tag = "full" if TRAIN_CHAR_LIMIT is None else str(TRAIN_CHAR_LIMIT)
val_limit_tag = "full" if VAL_CHAR_LIMIT is None else str(VAL_CHAR_LIMIT)
cache_name = f"bpe{len(tokenizer.id_to_token)}_{bos_tag}_train{train_limit_tag}_val{val_limit_tag}.pt"
cache_path = cache_root / cache_name

encode_start_time = time.perf_counter()
if cache_path.exists():
    cached = torch.load(cache_path, map_location="cpu")
    train_ids = cached["train_ids"]
    val_ids = cached["val_ids"]
    print("loaded token id cache:", cache_path)
else:
    train_ids = encode_lm_text(train_text, encode_each_line_with_bos_eos)
    val_ids = encode_lm_text(val_text, encode_each_line_with_bos_eos)
    torch.save({"train_ids": train_ids, "val_ids": val_ids}, cache_path)
    print("saved token id cache:", cache_path)
encode_elapsed = time.perf_counter() - encode_start_time

train_loader = create_dataloader(
    train_ids,
    context_length=context_length,
    batch_size=batch_size,
    shuffle=True,
)
val_loader = create_dataloader(
    val_ids,
    context_length=context_length,
    batch_size=batch_size,
    shuffle=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("train chars:", len(train_text), "val chars:", len(val_text))
print("train tokens:", len(train_ids), "val tokens:", len(val_ids))
print("train batches:", len(train_loader), "val batches:", len(val_loader))
print("config:", config)
print("batch_size:", batch_size, "learning_rate:", learning_rate, "num_epochs:", num_epochs)
print("encode_each_line_with_bos_eos:", encode_each_line_with_bos_eos)
print("encode_time_sec:", round(encode_elapsed, 2))
print("encode_time_min:", round(encode_elapsed / 60, 2))

model = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

start_time = time.perf_counter()
train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=eval_freq,
    eval_iter=eval_iter,
    start_context="이 영화는",
    tokenizer=tokenizer,
    ckpt_freq=ckpt_freq,
)
elapsed = time.perf_counter() - start_time

print("training_loop_time_sec:", round(elapsed, 2))
print("training_loop_time_min:", round(elapsed / 60, 2))
final_full_val_loss = calc_loss_loader(val_loader, model, device, num_batches=None)
total_elapsed = time.perf_counter() - cell_start_time

print("best_train_loss:", min(train_losses) if train_losses else None)
print("best_val_loss_estimate:", min(val_losses) if val_losses else None)
print("final_train_loss_estimate:", train_losses[-1] if train_losses else None)
print("final_val_loss_estimate:", val_losses[-1] if val_losses else None)
print("final_full_val_loss:", final_full_val_loss)
print("total_cell_time_sec:", round(total_elapsed, 2))
print("total_cell_time_min:", round(total_elapsed / 60, 2))

eval_steps = [(i + 1) * eval_freq for i in range(len(train_losses))]
eval_epochs = [((step - 1) // len(train_loader)) + 1 for step in eval_steps]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(eval_steps, train_losses, marker="o", label="Train")
ax.plot(eval_steps, val_losses, marker="o", label="Val")
ax.set_xlabel("Global step / Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training / Validation Loss")
ax.legend()
ax.grid(alpha=0.25)

if eval_steps:
    tick_count = min(8, len(eval_steps))
    tick_indices = sorted(set(round(i * (len(eval_steps) - 1) / max(1, tick_count - 1)) for i in range(tick_count)))
    tick_values = [eval_steps[i] for i in tick_indices]
    tick_labels = [f"{eval_steps[i]}\nE{eval_epochs[i]}" for i in tick_indices]
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels)

fig.tight_layout()
local_plot_path = Path("loss_curve.png")
fig.savefig(local_plot_path, dpi=150, bbox_inches="tight")
plt.show()

# Colab 런타임 저장소의 checkpoints/는 사라질 수 있으므로 Drive에도 복사합니다.
drive_root = Path("/content/drive/MyDrive/W14-gpt-lab")
if drive_root.exists():
    bos_tag = "bos" if encode_each_line_with_bos_eos else "nobos"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = (
        f"{timestamp}_pretrain"
        f"_ctx{context_length}"
        f"_{bos_tag}"
        f"_emb{config['emb_dim']}"
        f"_heads{config['n_heads']}"
        f"_layers{config['n_layers']}"
        f"_drop{config['drop_rate']}"
        f"_bs{batch_size}"
        f"_lr{learning_rate:g}"
        f"_ep{num_epochs}"
    )
    drive_run_dir = drive_root / "runs" / run_name

    suffix = 1
    while drive_run_dir.exists():
        drive_run_dir = drive_root / "runs" / f"{run_name}_repeat{suffix}"
        suffix += 1

    drive_ckpt_dir = drive_run_dir / "checkpoints"
    drive_ckpt_dir.mkdir(parents=True, exist_ok=False)

    drive_plot_path = drive_run_dir / "loss_curve.png"
    if local_plot_path.exists():
        shutil.copy2(local_plot_path, drive_plot_path)

    for epoch in range(1, num_epochs + 1):
        ckpt_path = Path("checkpoints") / f"ckpt_epoch_{epoch}.pt"
        if ckpt_path.exists():
            shutil.copy2(ckpt_path, drive_ckpt_dir / ckpt_path.name)

    summary = {
        "run_name": run_name,
        "run_dir": str(drive_run_dir),
        "train_chars": len(train_text),
        "val_chars": len(val_text),
        "train_tokens": len(train_ids),
        "val_tokens": len(val_ids),
        "token_cache_path": str(cache_path),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
        "config": config,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "eval_freq": eval_freq,
        "eval_iter": eval_iter,
        "ckpt_freq": ckpt_freq,
        "encode_each_line_with_bos_eos": encode_each_line_with_bos_eos,
        "encode_time_sec": round(encode_elapsed, 2),
        "training_loop_time_sec": round(elapsed, 2),
        "total_cell_time_sec": round(total_elapsed, 2),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "eval_steps": eval_steps,
        "eval_epochs": eval_epochs,
        "loss_curve_path": str(drive_plot_path),
        "best_train_loss": min(train_losses) if train_losses else None,
        "best_val_loss_estimate": min(val_losses) if val_losses else None,
        "final_train_loss_estimate": train_losses[-1] if train_losses else None,
        "final_val_loss_estimate": val_losses[-1] if val_losses else None,
        "final_full_val_loss": final_full_val_loss,
    }

    with open(drive_run_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("saved run summary:", drive_run_dir / "summary.json")
    print("saved loss curve:", drive_plot_path)
    print("saved checkpoints:", drive_ckpt_dir)
else:
    print("Drive path not found. Checkpoints remain in local ./checkpoints")
